Today's topics:
* slicing and indexing arrays
* array stacking, shape, and axis

# Moving from rows to tables

## Same chemistry, lower cost

Lithium-ion cells are in everything, and lithium is scarce and expensive.
Sodium sits directly below lithium in group 1, with the same single valence
electron, so a sodium-ion cell runs on the same idea. Sodium is also one of the
most abundant elements on Earth.

Battery makers have been building sodium-ion batteries on exactly that
principle. The cells work, but they store less energy per kilogram.
Why? We'll find out.

## Periodic table has trends

Last class we pulled single elements out of the periodic table, one index at a
time. That works fine, but it isn't how chemists think about the periodic table.

<img src="https://upload.wikimedia.org/wikipedia/commons/a/ad/Comparative_atomic_sizes.png" height="350">

*Relative atomic sizes, transition metals omitted. CK-12 Foundation,
[CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/), via
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Comparative_atomic_sizes.png),
unmodified.*

Chemists think in **rows and columns**. 

Period 2 is a row: Li through F, and the atoms shrink from left to right.

Group 1 is a column: H, Li, Na, K. The atoms grow from top to bottom.

Neither of those correspond to a scalar index. The column isn't even a contiguous stretch of the array!

We'll load the same dataset as last class, data from the periodic table.
Run the cell below; you don't need to read its
code. It yields five parallel arrays, ordered by atomic number.

In [1]:
import numpy as np

In [2]:
#@title Load the real periodic-table dataset (click ▶ to run, data loading, not a learning objective) { display-mode: "form" }
import os

import numpy as np
import pandas as pd

_file = 'mendeleev_elements.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}',
                 f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    _table = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

symbol = _table['symbol'].to_numpy()
atomic_number = _table['atomic_number'].to_numpy()
radius_pm = _table['covalent_radius_cordero'].to_numpy()
density = _table['density'].to_numpy()
atomic_weight = _table['atomic_weight'].to_numpy()

print(f'Loaded {len(symbol)} elements from {_file}')

Loaded 118 elements from mendeleev_elements.csv


# Slicing: a range of entries

One element at a time is slow going. `array[start:stop]` returns a whole range
at once, from `start` up to but **not including** `stop`.

## Slice notation

That `:` is worth a closer look, because we will use it constantly.

We often want more than one value at a time, and the `:` operator indicates a
range of values. For instance,

> `0:3` indicates elements `0, 1, 2`

Note the way this excludes the value on the right. 

If you want to include all values you can just use `:`, since Python interprets the empty limits as no
limits to the range.

The `:` is called **slice notation**.

In [3]:
print(symbol[0:4])     # H, He, Li, Be
print(symbol[:4])      # the same four: an empty start means "from the beginning"
print(symbol[1:3])     # He, Li
print(symbol[110:])    # an empty stop means "to the end"

['H' 'He' 'Li' 'Be']
['H' 'He' 'Li' 'Be']
['He' 'Li']
['Rg' 'Cn' 'Nh' 'Fl' 'Mc' 'Lv' 'Ts' 'Og']


Let's investigate that `stop` index closely.
We asked for `2:9` and neon, index 9, is not in the result.

The slice length is exactly `stop - start`, because *the stop position itself is excluded.*

Period 2 runs from Li through F, indices 2 through 8.
Ne will get left off the end; the noble gases don't follow this trend.

In [4]:
print(symbol[2:9])
print(radius_pm[2:9])

['Li' 'Be' 'B' 'C' 'N' 'O' 'F']
[128.  96.  84.  73.  71.  66.  57.]


`radius_pm` is the covalent radius in picometers, so these numbers say how big
each atom is when it bonds.

They shrink steadily all the way across.

Moving right you add protons, the nuclear charge pulls harder on
the same electron shell, and the atom contracts.

Now period 3, Na through Cl, indices 10 through 16:

In [5]:
print(symbol[10:17])
print(radius_pm[10:17])

['Na' 'Mg' 'Al' 'Si' 'P' 'S' 'Cl']
[166. 141. 121. 111. 107. 105. 102.]


The same contraction, but every atom is bigger than its period-2 counterpart,
because period 3 fills a shell that sits farther out.

Leave out `start` or `stop` to mean "from the beginning" or "to the end", and add a
third number for a step:

In [6]:
print(symbol[:5])        # first five
print(symbol[-5:])       # last five
print(symbol[0:20:2])    # every other element

['H' 'He' 'Li' 'Be' 'B']
['Fl' 'Mc' 'Lv' 'Ts' 'Og']
['H' 'Li' 'B' 'N' 'F' 'Na' 'Al' 'P' 'Cl' 'K']


The third number is the step. A negative one traverses the range backward:

In [7]:
print(symbol[1:8:2])     # every 2nd element starting with the one with "1" index
print(symbol[8:0:-1])    # backward from 8 down to 1
print(symbol[7::-1])     # the first eight, reversed
print(symbol[::10])      # every tenth element

['He' 'Be' 'C' 'O']
['F' 'O' 'N' 'C' 'B' 'Be' 'Li' 'He']
['O' 'N' 'C' 'B' 'Be' 'Li' 'He' 'H']
['H' 'Na' 'Sc' 'Ga' 'Nb' 'Sb' 'Pm' 'Lu' 'Tl' 'Pa' 'Md' 'Rg']


# Indexing arbitrary positions

Slicing works when the entries you want sit next to each other in a contiguous group.

Group 1 does not: hydrogen, lithium, sodium, potassium land at 0, 2, 10, 18.
That's not a valid range for the slicing operator.

For this, put a *collection of indices* inside the square brackets:

In [8]:
group_1 = [0, 2, 10, 18]
print(symbol[group_1])
print(radius_pm[group_1])

['H' 'Li' 'Na' 'K']
[ 31. 128. 166. 203.]


The square brackets assigned to `group_1` make a Python `list`.
For now, just treat that as a recipe for collecting indices; we'll study it later in detail.

The entries come back in the order you asked for, and a position may repeat.
This would be difficult with slice notation, because `:` can't slice at
irregular intervals.

In [9]:
print(symbol[[1, 0, 3]])    # your order, not sorted
print(symbol[[2, 2, 2]])    # a position may repeat
print(symbol[[0, -1]])      # first and last

['He' 'H' 'Be']
['Li' 'Li' 'Li']
['H' 'Og']


<img src="https://upload.wikimedia.org/wikipedia/commons/a/ad/Comparative_atomic_sizes.png" height="260">

*Same figure as above. CK-12 Foundation, CC BY-SA 3.0.*

Find hydrogen at the top of the first column. It sits there because it has one
valence electron, but it isn't an alkali metal and it has no inner shells, so at
31 pm it's far smaller than the lithium below it. We can see that in the array.

In [10]:
print(radius_pm[[0, 2, 10, 18]])    # H, Li, Na, K

[ 31. 128. 166. 203.]


The positions 0, 2, 10, 18 are separated by 2, 8, and 8. Those are the lengths
of the first three periods, so group 1 lands at 0, 2, 10, and 18 in the flat
array.

We see that the radius grows down the column, within group 1.

### [Check your understanding]

You have two new tools: a slice, and a list of positions (fancy indexing).

Period 4 starts at potassium, index 18, and runs to krypton at index 35. As
before, leave the noble gas off the end. Group 2 is the alkaline earth metals,
and beryllium, magnesium, calcium, and strontium sit at indices 3, 11, 19, 37.

In the code cell below:

1. Slice the symbols and covalent radii for period 4, potassium through bromine,
   and print both
2. Fancy-index the symbols and radii for those four group 2 elements, and print
   both
3. Period 4 doesn't contract as smoothly as period 2 did. Find every place
   where the radius stops shrinking, and name the elements

*Optional challenge*

4. Fancy-index the halogen radii

# 2D arrays

When two arrays line up like our period 1 and period 2 arrays, we can stack them into a **2-D array**, a
table with rows and columns:

In [11]:
period_2 = radius_pm[2:9]
period_3 = radius_pm[10:17]

radii = np.array([period_2, period_3])
print(radii)

[[128.  96.  84.  73.  71.  66.  57.]
 [166. 141. 121. 111. 107. 105. 102.]]


One important thing about this table: every number in it is a covalent radius
in picometers. Same property, same units, throughout.

We stacked two of the same thing, not two different properties.

Stacking a temperature on top of a length would
produce a table we could not do (meaningful) arithmetic on.

In [12]:
print(radii.shape)

(2, 7)


The `shape` attribute tells us the (rows, columns) of the `array`. Here it
reports `(2, 7)`: 2 rows, 7 columns. Row 0 is period 2, row 1 is period 3, and
column `j` is one group position in both.

The rows have to be the same length. Forget to drop Ar and period 3 is 8 long
against period 2's 7, which raises an error rather than building a "ragged"
table:

In [13]:
# EXPECTED-ERROR: this cell intentionally fails
np.array([radius_pm[2:9], radius_pm[10:18]])
#
# Notice that this one is written differently from what we have been doing.
# 
#  With
# `np.array(...)` or `np.min(...)` you pass the array to a NumPy function. 
# 
# With `radii.shape` you attach the name to the array itself with a dot, and you get a property of that array.
# 
# Anything attached with a dot belongs to that specific
# array. And there are no parentheses on `.shape` because you're reading a fact (*attribute*)
# off the array rather than calling a function (*method*).

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

## 2D indexing: `array[row, col]`

With two dimensions you give two numbers, row first:

In [14]:
print(radii[0, 0])    # period 2, first group position: lithium
print(radii[1, 0])    # period 3, first group position: sodium

128.0
166.0


Use `:` to grab a whole row or a whole column:

In [15]:
print(radii[0, :])    # all of period 2
print(radii[:, 0])    # the group 1 column, both periods
print(radii[:, 1:3])  # groups 2 and 3, both periods

[128.  96.  84.  73.  71.  66.  57.]
[128. 166.]
[[ 96.  84.]
 [141. 121.]]


A single number for one dimension drops it from the result. A slice keeps it,
even when the slice is one entry wide:

In [16]:
print(radii[0, 0].shape)      # ()
print(radii[0:1, 0:1].shape)  # (1, 1)

()
(1, 1)


## Choosing a direction with `axis`

`np.max` computes the maximum value. On a 2-D array you have to say which direction to
find the maximum over, and which direction you want depends on what you are asking about
the materials.

This keyword argument indicates which *direction* the maximum should be taken
over. So `axis=0` indicates taking the maximum over a column, collecting values
from the up-down direction:

In [17]:
print(np.max(radii, axis=0))

[166. 141. 121. 111. 107. 105. 102.]


`axis=0` collapses down the columns. Seven numbers come back, one per group
position. The period distinction is gone, and the seven numbers still line up by
group.

In [18]:
print(np.max(radii, axis=1))

[128. 166.]


`axis=1` collapses across the rows. Two numbers come back: the max radius of
lithium through fluorine and the max radius of sodium through chlorine. Now the
group distinction is gone, and the two numbers still line up by period.

To keep this straight, look at what is left over. `axis=0` collapsed the periods
and left seven group positions. `axis=1` collapsed the groups and left two
periods.

You can think of this operation as eliminating the shape in that spot of
`shape`.

 With `axis=0` we get a `(7,)`-shaped result, eliminating `shape[0]`,
which is 2. 

With `axis=1` we get a `(2,)`-shaped result, eliminating
`shape[1]`, which is 7:

In [19]:
print(radii.shape)
print(np.max(radii, axis=0).shape)
print(np.max(radii, axis=1).shape)

(2, 7)
(7,)
(2,)


`np.min`, `np.max`, and `np.sum`, and other common functions all take `axis` the same way.

The CSV behind today's arrays came out of a Python package called `mendeleev`,
with 118 elements and dozens of properties for each one. You don't need this
syntax and it won't be on anything. I bring it up so you know it's there:

In [20]:
#@title Optional tool glimpse: the whole table in one package (click ▶ to run) { display-mode: "form" }
try:
    import mendeleev
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mendeleev==1.1.0"])
    import mendeleev

In [21]:
fe = mendeleev.element("Fe")
display(fe)

<Element(abundance_crust=56300.0, abundance_sea=0.002, atomic_number=26, atomic_radius=140.0, atomic_radius_rahm=237.0, atomic_weight=55.845, atomic_weight_uncertainty=0.002, block='d', c6=482.0, c6_gb=548.0, cas='7439-89-6', covalent_radius_bragg=140.0, covalent_radius_cordero=142.0, covalent_radius_pyykko=115.99999999999999, covalent_radius_pyykko_double=109.0, covalent_radius_pyykko_triple=102.0, cpk_color='#ffa500', density=7.87, description="Silvery malleable and ductile metallic transition element. Has nine isotopes and is the fourth most abundant element in the earth's crust. Required by living organisms as a trace element (used in hemoglobin in humans.) Quite reactive, oxidizes in moist air, displaces hydrogen from dilute acids and combines with nonmetallic elements.", dipole_polarizability=62.0, dipole_polarizability_unc=4.0, discoverers='Known to the ancients.', discovery_location=None, discovery_year=None, ec=<ElectronicConfiguration(conf="1s2 2s2 2p6 3s2 3p6 3d6 4s2")>, eco

## Back to the batteries

Group 1 is the column we pulled out at the start. `atomic_weight` is one of the
five arrays the setup cell loaded, so we can read the answer in the same way:

In [22]:
print(symbol[group_1])
print(atomic_weight[group_1])

['H' 'Li' 'Na' 'K']
[ 1.008       6.94       22.98976928 39.0983    ]


Lithium weighs 6.94, sodium 22.99. Swapping lithium for sodium keeps the
chemistry, because they share a column, has three times the mass
for every charge carrier you move. So, a lot less energy storage per kilogram.

The radius trend matters too. A bigger ion needs a different host material to
accommodate it, so sodium cells can't use the same graphite anode.

# Quiz warmup

1. f-strings from [L02](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture02.ipynb)
- How would you write a complete f-string to display a float?
2. Arrays and indexing from [L03](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture03.ipynb)
- How are the different arrays we use related? How does indexing allow different arrays to represent the same subject (elements)?

## Further reading

- [VanderPlas: The basics of NumPy arrays](https://jakevdp.github.io/PythonDataScienceHandbook/02.02-the-basics-of-numpy-arrays.html) (slicing and reshaping)
- [VanderPlas: Fancy indexing](https://jakevdp.github.io/PythonDataScienceHandbook/02.07-fancy-indexing.html)
- [VanderPlas: Aggregations](https://jakevdp.github.io/PythonDataScienceHandbook/02.04-computation-on-arrays-aggregates.html) (the `axis` argument)
- [Cordero et al., "Covalent radii revisited" (2008)](https://doi.org/10.1039/B801115J), the source of the radii used today